# 28 — Supervised Baseline

## Objetivo

Construir el **baseline supervisado** únicamente para los targets que el notebook 27e declaró evaluables con soporte mínimo de clases y grupos.

Con el estado actual del master v2:

```text
H1 → bloqueado
H4 → bloqueado
H5 → bloqueado
H6 → BASELINE_CANDIDATE
```

Por tanto, este notebook debe entrenar y evaluar **solo H6** mientras los otros targets permanezcan bloqueados.

La finalidad científica del baseline es establecer una referencia contra la cual comparar posteriormente el aprendizaje semisupervisado:

$$
\text{Supervised baseline}(L)
\quad vs \quad
\text{Semi-supervised}(L \cup U)
$$

Reglas:

- no usar `U` durante el entrenamiento supervisado;
- no usar `source_dataset`, `source_match_level` ni procedencia como predictores;
- no hacer tuning usando los folds de evaluación;
- usar las mismas divisiones agrupadas posteriormente en el notebook 29;
- `1 = violación`, `0 = no violación`;
- reportar explícitamente el tamaño muy pequeño del conjunto etiquetado.


## 1. Configuración

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import warnings

from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedGroupKFold

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix
)

import joblib

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", 500)

PROJECT_DIR = Path("..").resolve()

SEMISUP_INTERIM_DIR = (
    PROJECT_DIR
    / "data"
    / "semisupervised"
    / "interim"
)

SEMISUP_REPORTS_DIR = (
    PROJECT_DIR
    / "reports"
    / "tables"
    / "semisupervised"
)

MODEL_DIR = (
    PROJECT_DIR
    / "models"
    / "semisupervised"
    / "baseline"
)

for directory in [
    SEMISUP_REPORTS_DIR,
    MODEL_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

FEATURES = [
    "structure_depth",
    "interactive_control_count",
    "interactive_label_ratio"
]

ALL_TARGETS = [
    "H1_violation",
    "H4_violation",
    "H5_violation",
    "H6_violation"
]

RANDOM_STATE = 42
PREDICTION_THRESHOLD = 0.50

print("PROJECT_DIR:", PROJECT_DIR)


PROJECT_DIR: /Users/uriel/Documents/machine_learning


## 2. Cargar master v2 y decisión de trainability

In [2]:
master_path = (
    SEMISUP_INTERIM_DIR
    / "semisupervised_proxy_master_dataset_v2.csv"
)

readiness_path = (
    SEMISUP_REPORTS_DIR
    / "semisupervised_master_v2_target_readiness.csv"
)

decision_path = (
    SEMISUP_REPORTS_DIR
    / "semisupervised_master_v2_salvage_decision.csv"
)

required_paths = [
    master_path,
    readiness_path,
    decision_path
]

missing = [
    str(path)
    for path in required_paths
    if not path.exists()
]

assert missing == [], missing

master = pd.read_csv(
    master_path
)

readiness = pd.read_csv(
    readiness_path
)

decision = pd.read_csv(
    decision_path
)

print("Master:", master.shape)

display(readiness)
display(decision)


Master: (1012, 23)


,target,n_rows_catching,n_labeled,n_unknown,n_0,n_1,n_classes,both_classes_observed,binary_seed_identifiable,grouped_holdout_class_support,baseline_status
0,H1_violation,12,12,0,0,12,1,False,False,False,NOT_TRAINABLE
1,H4_violation,12,11,1,1,10,2,True,True,False,BINARY_SEED_ONLY_HOLDOUT_BLOCKED
2,H5_violation,12,12,0,1,11,2,True,True,False,BINARY_SEED_ONLY_HOLDOUT_BLOCKED
3,H6_violation,12,12,0,9,3,2,True,True,True,BASELINE_CANDIDATE


,master_v2_built,master_v2_rows,salvaged_negative_feature_rows,excluded_incomplete_negative_rows,ready_targets,blocked_targets,critical_excluded_sites,full_baseline_ready,global_status
0,True,1012,10,2,H6_violation,H1_violation | H4_violation | H5_violation,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,False,PROCEED_TO_28_PARTIAL_BASELINE


## 3. Resolver targets habilitados

Solo se consideran targets con:

```text
grouped_holdout_class_support = True
```

No se fuerza el entrenamiento de H1/H4/H5.


In [3]:
def to_bool(series):
    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("true")
    )


readiness = readiness.copy()

if (
    readiness[
        "grouped_holdout_class_support"
    ].dtype
    !=
    bool
):

    readiness[
        "grouped_holdout_class_support"
    ] = to_bool(
        readiness[
            "grouped_holdout_class_support"
        ]
    )

ready_targets = (
    readiness.loc[
        readiness[
            "grouped_holdout_class_support"
        ],
        "target"
    ]
    .astype(str)
    .tolist()
)

blocked_targets = [
    target
    for target in ALL_TARGETS
    if target not in ready_targets
]

print(
    "READY TARGETS:",
    ready_targets
)

print(
    "BLOCKED TARGETS:",
    blocked_targets
)

assert len(ready_targets) > 0

# Con el master actual esperamos solo H6.
if ready_targets != [
    "H6_violation"
]:
    warnings.warn(
        "El conjunto de targets listos cambió desde 27e. "
        "El notebook procesará los targets habilitados realmente."
    )


READY TARGETS: ['H6_violation']
BLOCKED TARGETS: ['H1_violation', 'H4_violation', 'H5_violation']


## 4. Construir L supervisado por target

Una fila entra a `L_target` únicamente cuando:

- las tres X están presentes;
- Y del target es 0 o 1;
- existe `group_id`.

`source_match_level` se conserva solo para auditoría.


In [4]:
labeled_sets = {}

labeled_summary_records = []

for target in ready_targets:

    subset = master[
        master[
            target
        ].notna()
    ].copy()

    for feature in FEATURES:
        subset[
            feature
        ] = pd.to_numeric(
            subset[
                feature
            ],
            errors="coerce"
        )

    subset[
        target
    ] = pd.to_numeric(
        subset[
            target
        ],
        errors="coerce"
    )

    subset = subset[
        subset[
            target
        ].isin(
            [
                0,
                1
            ]
        )
        &
        subset[
            FEATURES
        ].notna().all(
            axis=1
        )
        &
        subset[
            "group_id"
        ].notna()
    ].copy()

    labeled_sets[
        target
    ] = subset

    labeled_summary_records.append({
        "target":
            target,

        "rows":
            len(
                subset
            ),

        "groups":
            subset[
                "group_id"
            ].nunique(),

        "n_0":
            int(
                (
                    subset[
                        target
                    ]
                    ==
                    0
                ).sum()
            ),

        "n_1":
            int(
                (
                    subset[
                        target
                    ]
                    ==
                    1
                ).sum()
            )
    })

labeled_summary = pd.DataFrame(
    labeled_summary_records
)

display(
    labeled_summary
)


,target,rows,groups,n_0,n_1
0,H6_violation,12,12,9,3


## 5. Auditoría de procedencia

`source_match_level` **no será predictor**.

Se reporta porque el seed H6 mezcla:

- filas `A_CONFIRMED_STRONG`;
- filas `B_MODERATE_VALIDATED`;
- filas `C_EXACT_NAME_TECHNICAL`.

Si la distribución de procedencia está asociada a la clase, eso debe tratarse como una limitación de validez.


In [5]:
provenance_records = []

for target in ready_targets:

    df = labeled_sets[
        target
    ]

    if (
        "source_match_level"
        not in df.columns
    ):

        continue

    table = (
        df
        .groupby(
            [
                "source_match_level",
                target
            ],
            dropna=False
        )
        .size()
        .reset_index(
            name="n"
        )
    )

    table[
        "target"
    ] = target

    table = table.rename(
        columns={
            target:
                "observed_y"
        }
    )

    provenance_records.append(
        table
    )

provenance_by_class = (
    pd.concat(
        provenance_records,
        ignore_index=True
    )
    if provenance_records
    else pd.DataFrame()
)

display(
    provenance_by_class
)


,source_match_level,observed_y,n,target
0,A_CONFIRMED_STRONG,1.0,2,H6_violation
1,B_MODERATE_VALIDATED,0.0,3,H6_violation
2,C_EXACT_NAME_TECHNICAL,0.0,6,H6_violation
3,C_EXACT_NAME_TECHNICAL,1.0,1,H6_violation


## 6. Construir folds agrupados y estratificados

Se usa `StratifiedGroupKFold`.

Para H6 hay tres positivos observados, por lo que el máximo razonable para mantener representación de la clase positiva es:

```text
n_splits = min(3, n_0, n_1)
```

La asignación de folds se congela en CSV y será reutilizada en el notebook 29.

No se genera un split aleatorio diferente para el modelo semisupervisado.


In [6]:
fold_registry_records = []

fold_assignments = {}

for target in ready_targets:

    df = labeled_sets[
        target
    ].reset_index(
        drop=True
    )

    y = df[
        target
    ].astype(
        int
    ).to_numpy()

    groups = df[
        "group_id"
    ].astype(
        str
    ).to_numpy()

    n_0 = int(
        (
            y
            ==
            0
        ).sum()
    )

    n_1 = int(
        (
            y
            ==
            1
        ).sum()
    )

    n_splits = min(
        3,
        n_0,
        n_1
    )

    assert n_splits >= 2

    splitter = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    assignment = np.full(
        len(df),
        -1,
        dtype=int
    )

    split_pairs = []

    for fold_id, (
        train_idx,
        test_idx
    ) in enumerate(
        splitter.split(
            df[
                FEATURES
            ],
            y,
            groups
        )
    ):

        assignment[
            test_idx
        ] = fold_id

        split_pairs.append(
            (
                train_idx,
                test_idx
            )
        )

        train_groups = set(
            groups[
                train_idx
            ]
        )

        test_groups = set(
            groups[
                test_idx
            ]
        )

        assert (
            train_groups
            .isdisjoint(
                test_groups
            )
        )

        fold_registry_records.append({
            "target":
                target,

            "fold_id":
                fold_id,

            "train_rows":
                len(
                    train_idx
                ),

            "test_rows":
                len(
                    test_idx
                ),

            "train_n_0":
                int(
                    (
                        y[
                            train_idx
                        ]
                        ==
                        0
                    ).sum()
                ),

            "train_n_1":
                int(
                    (
                        y[
                            train_idx
                        ]
                        ==
                        1
                    ).sum()
                ),

            "test_n_0":
                int(
                    (
                        y[
                            test_idx
                        ]
                        ==
                        0
                    ).sum()
                ),

            "test_n_1":
                int(
                    (
                        y[
                            test_idx
                        ]
                        ==
                        1
                    ).sum()
                )
        })

    assert (
        assignment
        >=
        0
    ).all()

    df[
        "evaluation_fold"
    ] = assignment

    fold_assignments[
        target
    ] = {
        "data":
            df,

        "splits":
            split_pairs,

        "n_splits":
            n_splits
    }

fold_registry = pd.DataFrame(
    fold_registry_records
)

display(
    fold_registry
)


,target,fold_id,train_rows,test_rows,train_n_0,train_n_1,test_n_0,test_n_1
0,H6_violation,0,8,4,6,2,3,1
1,H6_violation,1,8,4,6,2,3,1
2,H6_violation,2,8,4,6,2,3,1


## 7. Validar cada fold antes del entrenamiento

Cada fold de entrenamiento debe contener ambas clases.

Cuando sea posible, también se comprueba que cada test fold tenga ambas clases para permitir métricas por fold.


In [7]:
fold_validation = fold_registry.copy()

fold_validation[
    "train_has_both_classes"
] = (
    (
        fold_validation[
            "train_n_0"
        ]
        >
        0
    )
    &
    (
        fold_validation[
            "train_n_1"
        ]
        >
        0
    )
)

fold_validation[
    "test_has_both_classes"
] = (
    (
        fold_validation[
            "test_n_0"
        ]
        >
        0
    )
    &
    (
        fold_validation[
            "test_n_1"
        ]
        >
        0
    )
)

display(
    fold_validation
)

assert (
    fold_validation[
        "train_has_both_classes"
    ]
    .all()
)


,target,fold_id,train_rows,test_rows,train_n_0,train_n_1,test_n_0,test_n_1,train_has_both_classes,test_has_both_classes
0,H6_violation,0,8,4,6,2,3,1,True,True
1,H6_violation,1,8,4,6,2,3,1,True,True
2,H6_violation,2,8,4,6,2,3,1,True,True


## 8. Modelos baseline

Se comparan:

### Dummy baseline
`DummyClassifier(strategy="prior")`

Sirve para medir cuánto aporta el modelo respecto a la prevalencia de clase.

### Random Forest supervisado
Configuración fija, sin búsqueda de hiperparámetros:

```text
n_estimators = 500
class_weight = balanced_subsample
random_state = 42
```

No se optimiza el threshold: se mantiene `0.50`.


In [8]:
def make_models():

    return {
        "DUMMY_PRIOR":
            DummyClassifier(
                strategy="prior",
                random_state=RANDOM_STATE
            ),

        "RANDOM_FOREST":
            RandomForestClassifier(
                n_estimators=500,
                class_weight="balanced_subsample",
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
    }


## 9. Función de métricas

In [9]:
def safe_roc_auc(
    y_true,
    y_prob
):

    if (
        np.unique(
            y_true
        ).size
        <
        2
    ):
        return np.nan

    return roc_auc_score(
        y_true,
        y_prob
    )


def compute_metrics(
    y_true,
    y_pred,
    y_prob
):

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[
            0,
            1
        ]
    )

    tn, fp, fn, tp = (
        cm.ravel()
    )

    return {
        "n":
            len(
                y_true
            ),

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),

        "precision_1":
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "recall_1":
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "f1_1":
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "mcc":
            matthews_corrcoef(
                y_true,
                y_pred
            ),

        "roc_auc":
            safe_roc_auc(
                y_true,
                y_prob
            ),

        "pr_auc":
            average_precision_score(
                y_true,
                y_prob
            ),

        "brier":
            brier_score_loss(
                y_true,
                y_prob
            ),

        "tn":
            int(
                tn
            ),

        "fp":
            int(
                fp
            ),

        "fn":
            int(
                fn
            ),

        "tp":
            int(
                tp
            )
    }


## 10. Predicciones out-of-fold

Cada fila etiquetada se predice únicamente con un modelo que **no fue entrenado con esa fila**.

Las probabilidades OOF se guardan para que el notebook 30 pueda reutilizarlas.


In [10]:
oof_records = []
fold_metric_records = []

for target in ready_targets:

    info = fold_assignments[
        target
    ]

    df = info[
        "data"
    ]

    X = df[
        FEATURES
    ].astype(
        float
    )

    y = df[
        target
    ].astype(
        int
    ).to_numpy()

    for model_name in [
        "DUMMY_PRIOR",
        "RANDOM_FOREST"
    ]:

        oof_prob = np.full(
            len(
                df
            ),
            np.nan,
            dtype=float
        )

        oof_pred = np.full(
            len(
                df
            ),
            -1,
            dtype=int
        )

        for fold_id, (
            train_idx,
            test_idx
        ) in enumerate(
            info[
                "splits"
            ]
        ):

            model = make_models()[
                model_name
            ]

            model.fit(
                X.iloc[
                    train_idx
                ],
                y[
                    train_idx
                ]
            )

            prob = model.predict_proba(
                X.iloc[
                    test_idx
                ]
            )[
                :,
                list(
                    model.classes_
                ).index(
                    1
                )
            ]

            pred = (
                prob
                >=
                PREDICTION_THRESHOLD
            ).astype(
                int
            )

            oof_prob[
                test_idx
            ] = prob

            oof_pred[
                test_idx
            ] = pred

            fold_metrics = compute_metrics(
                y[
                    test_idx
                ],
                pred,
                prob
            )

            fold_metric_records.append({
                "target":
                    target,

                "model":
                    model_name,

                "fold_id":
                    fold_id,

                **fold_metrics
            })

        assert (
            ~np.isnan(
                oof_prob
            )
        ).all()

        assert (
            oof_pred
            >=
            0
        ).all()

        for idx in range(
            len(
                df
            )
        ):

            oof_records.append({
                "target":
                    target,

                "model":
                    model_name,

                "master_instance_id":
                    df.loc[
                        idx,
                        "master_instance_id"
                    ],

                "group_id":
                    df.loc[
                        idx,
                        "group_id"
                    ],

                "source_match_level":
                    df.loc[
                        idx,
                        "source_match_level"
                    ]
                    if
                    "source_match_level"
                    in
                    df.columns
                    else "",

                "evaluation_fold":
                    int(
                        df.loc[
                            idx,
                            "evaluation_fold"
                        ]
                    ),

                "y_true":
                    int(
                        y[
                            idx
                        ]
                    ),

                "y_prob":
                    float(
                        oof_prob[
                            idx
                        ]
                    ),

                "y_pred":
                    int(
                        oof_pred[
                            idx
                        ]
                    )
            })

oof_predictions = pd.DataFrame(
    oof_records
)

fold_metrics = pd.DataFrame(
    fold_metric_records
)

display(
    oof_predictions
)

display(
    fold_metrics
)


,target,model,master_instance_id,group_id,source_match_level,evaluation_fold,y_true,y_prob,y_pred
0,H6_violation,DUMMY_PRIOR,catching_ux_flaws__startbootstrap_3_col_portfolio,catching__startbootstrap_3_col_portfolio,A_CONFIRMED_STRONG,2,1,0.250,0
1,H6_violation,DUMMY_PRIOR,catching_ux_flaws__startbootstrap_sb_admin_2,catching__startbootstrap_sb_admin_2,A_CONFIRMED_STRONG,1,1,0.250,0
2,H6_violation,DUMMY_PRIOR,catching_ux_flaws__beginner_html_site_styled,catching__beginner_html_site_styled,B_MODERATE_VALIDATED,2,0,0.250,0
3,H6_violation,DUMMY_PRIOR,catching_ux_flaws__conference_website_template,catching__conference_website_template,B_MODERATE_VALIDATED,2,0,0.250,0
4,H6_violation,DUMMY_PRIOR,catching_ux_flaws__personal_website_template,catching__personal_website_template,B_MODERATE_VALIDATED,1,0,0.250,0
5,H6_violation,DUMMY_PRIOR,catching_ux_flaws__charity_responsive_template,catching__charity_responsive_template,C_EXACT_NAME_TECHNICAL,0,0,0.250,0
6,H6_violation,DUMMY_PRIOR,catching_ux_flaws__simple_gallery_responsive_template,catching__simple_gallery_responsive_template,C_EXACT_NAME_TECHNICAL,2,0,0.250,0
7,H6_violation,DUMMY_PRIOR,catching_ux_flaws__single_page_small_business_web_application,catching__single_page_small_business_web_application,C_EXACT_NAME_TECHNICAL,1,0,0.250,0
8,H6_violation,DUMMY_PRIOR,catching_ux_flaws__startbootstrap_freelancer,catching__startbootstrap_freelancer,C_EXACT_NAME_TECHNICAL,0,0,0.250,0
9,H6_violation,DUMMY_PRIOR,catching_ux_flaws__startbootstrap_landing_page,catching__startbootstrap_landing_page,C_EXACT_NAME_TECHNICAL,0,0,0.250,0


,target,model,fold_id,n,accuracy,balanced_accuracy,precision_1,recall_1,f1_1,mcc,roc_auc,pr_auc,brier,tn,fp,fn,tp
0,H6_violation,DUMMY_PRIOR,0,4,0.75,0.5,0.0,0.0,0.0,0.0,0.500000,0.25,0.187500,3,0,1,0
1,H6_violation,DUMMY_PRIOR,1,4,0.75,0.5,0.0,0.0,0.0,0.0,0.500000,0.25,0.187500,3,0,1,0
2,H6_violation,DUMMY_PRIOR,2,4,0.75,0.5,0.0,0.0,0.0,0.0,0.500000,0.25,0.187500,3,0,1,0
3,H6_violation,RANDOM_FOREST,0,4,1.00,1.0,1.0,1.0,1.0,1.0,1.000000,1.00,0.166780,3,0,0,1
4,H6_violation,RANDOM_FOREST,1,4,0.75,0.5,0.0,0.0,0.0,0.0,0.000000,0.25,0.273806,3,0,1,0
5,H6_violation,RANDOM_FOREST,2,4,0.75,0.5,0.0,0.0,0.0,0.0,0.666667,0.50,0.169290,3,0,1,0


## 11. Métricas OOF agregadas

Estas métricas son la referencia principal del baseline.

Dado el tamaño extremadamente pequeño de `L`, deben interpretarse como **estimación experimental preliminar**, no como una estimación estable de desempeño poblacional.


In [11]:
aggregate_records = []

for (
    target,
    model_name
), group in (
    oof_predictions
    .groupby(
        [
            "target",
            "model"
        ]
    )
):

    metrics = compute_metrics(
        group[
            "y_true"
        ].to_numpy(),

        group[
            "y_pred"
        ].to_numpy(),

        group[
            "y_prob"
        ].to_numpy()
    )

    aggregate_records.append({
        "target":
            target,

        "model":
            model_name,

        "threshold":
            PREDICTION_THRESHOLD,

        **metrics
    })

aggregate_metrics = pd.DataFrame(
    aggregate_records
)

display(
    aggregate_metrics
)


,target,model,threshold,n,accuracy,balanced_accuracy,precision_1,recall_1,f1_1,mcc,roc_auc,pr_auc,brier,tn,fp,fn,tp
0,H6_violation,DUMMY_PRIOR,0.5,12,0.750000,0.500000,0.0,0.000000,0.0,0.000000,0.500000,0.250000,0.187500,9,0,3,0
1,H6_violation,RANDOM_FOREST,0.5,12,0.833333,0.666667,1.0,0.333333,0.5,0.522233,0.481481,0.511905,0.203292,9,0,2,1


## 12. Diferencia Random Forest vs Dummy

In [12]:
metric_columns = [
    "balanced_accuracy",
    "precision_1",
    "recall_1",
    "f1_1",
    "mcc",
    "roc_auc",
    "pr_auc",
    "brier"
]

comparison_records = []

for target in ready_targets:

    subset = aggregate_metrics[
        aggregate_metrics[
            "target"
        ]
        ==
        target
    ]

    rf = subset[
        subset[
            "model"
        ]
        ==
        "RANDOM_FOREST"
    ]

    dummy = subset[
        subset[
            "model"
        ]
        ==
        "DUMMY_PRIOR"
    ]

    if (
        rf.empty
        or
        dummy.empty
    ):
        continue

    rf = rf.iloc[
        0
    ]

    dummy = dummy.iloc[
        0
    ]

    record = {
        "target":
            target
    }

    for metric in metric_columns:

        record[
            f"rf_{metric}"
        ] = rf[
            metric
        ]

        record[
            f"dummy_{metric}"
        ] = dummy[
            metric
        ]

        if metric == "brier":

            record[
                f"rf_minus_dummy_{metric}"
            ] = (
                rf[
                    metric
                ]
                -
                dummy[
                    metric
                ]
            )

        else:

            record[
                f"rf_minus_dummy_{metric}"
            ] = (
                rf[
                    metric
                ]
                -
                dummy[
                    metric
                ]
            )

    comparison_records.append(
        record
    )

baseline_comparison = pd.DataFrame(
    comparison_records
)

display(
    baseline_comparison
)


,target,rf_balanced_accuracy,dummy_balanced_accuracy,rf_minus_dummy_balanced_accuracy,rf_precision_1,dummy_precision_1,rf_minus_dummy_precision_1,rf_recall_1,dummy_recall_1,rf_minus_dummy_recall_1,rf_f1_1,dummy_f1_1,rf_minus_dummy_f1_1,rf_mcc,dummy_mcc,rf_minus_dummy_mcc,rf_roc_auc,dummy_roc_auc,rf_minus_dummy_roc_auc,rf_pr_auc,dummy_pr_auc,rf_minus_dummy_pr_auc,rf_brier,dummy_brier,rf_minus_dummy_brier
0,H6_violation,0.666667,0.5,0.166667,1.0,0.0,1.0,0.333333,0.0,0.333333,0.5,0.0,0.5,0.522233,0.0,0.522233,0.481481,0.5,-0.018519,0.511905,0.25,0.261905,0.203292,0.1875,0.015792


## 13. Importancia de features

La importancia se calcula con un Random Forest final ajustado sobre todo `L_H6`.

No se interpreta como causalidad.

Este modelo final se guarda únicamente para reproducibilidad del baseline; el modelo candidato de despliegue se decidirá después del experimento semisupervisado.


In [13]:
feature_importance_records = []
final_model_registry = []

for target in ready_targets:

    df = labeled_sets[
        target
    ].copy()

    X = df[
        FEATURES
    ].astype(
        float
    )

    y = df[
        target
    ].astype(
        int
    )

    final_rf = RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    final_rf.fit(
        X,
        y
    )

    model_path = (
        MODEL_DIR
        /
        f"{target}_supervised_rf.joblib"
    )

    joblib.dump(
        final_rf,
        model_path
    )

    for feature, importance in zip(
        FEATURES,
        final_rf.feature_importances_
    ):

        feature_importance_records.append({
            "target":
                target,

            "feature":
                feature,

            "importance":
                float(
                    importance
                )
        })

    final_model_registry.append({
        "target":
            target,

        "model":
            "RandomForestClassifier",

        "model_path":
            str(
                model_path.relative_to(
                    PROJECT_DIR
                )
            ),

        "training_rows":
            len(
                df
            ),

        "feature_count":
            len(
                FEATURES
            )
    })

feature_importance = pd.DataFrame(
    feature_importance_records
)

final_model_registry = pd.DataFrame(
    final_model_registry
)

display(
    feature_importance.sort_values(
        [
            "target",
            "importance"
        ],
        ascending=[
            True,
            False
        ]
    )
)

display(
    final_model_registry
)


,target,feature,importance
1,H6_violation,interactive_control_count,0.380699
2,H6_violation,interactive_label_ratio,0.347799
0,H6_violation,structure_depth,0.271501


,target,model,model_path,training_rows,feature_count
0,H6_violation,RandomForestClassifier,models/semisupervised/baseline/H6_violation_supervised_rf.joblib,12,3


## 14. Auditoría de limitaciones del baseline

El notebook registra de forma explícita:

- targets bloqueados;
- tamaño de L;
- número de positivos/negativos;
- mezcla de niveles A/B/C;
- ausencia total de U en el entrenamiento;
- carácter experimental del espacio X proxy.


In [14]:
limitations_records = []

for target in ready_targets:

    df = labeled_sets[
        target
    ]

    limitations_records.extend([
        {
            "target":
                target,

            "issue":
                "VERY_SMALL_LABELED_SET",

            "value":
                len(
                    df
                ),

            "implication":
                (
                    "Metrics have high variance and "
                    "must be treated as preliminary."
                )
        },

        {
            "target":
                target,

            "issue":
                "PROXY_FEATURE_SPACE",

            "value":
                "EXPERIMENTAL_CONSTRUCT_PROXY",

            "implication":
                (
                    "Features are construct-level "
                    "proxies, not strict canonical Nielsen metrics."
                )
        },

        {
            "target":
                target,

            "issue":
                "MIXED_SOURCE_MATCH_LEVEL",

            "value":
                " | ".join(
                    sorted(
                        df[
                            "source_match_level"
                        ]
                        .dropna()
                        .astype(str)
                        .unique()
                        .tolist()
                    )
                ),

            "implication":
                (
                    "Observed labels include source matches "
                    "with different evidence strength."
                )
        },

        {
            "target":
                target,

            "issue":
                "UNLABELED_DATA_USED_IN_BASELINE",

            "value":
                False,

            "implication":
                (
                    "This is intentional; U is reserved "
                    "for the semi-supervised experiment."
                )
        }
    ])

for target in blocked_targets:

    limitations_records.append({
        "target":
            target,

        "issue":
            "TARGET_BLOCKED",

        "value":
            True,

        "implication":
            (
                "No supervised baseline is reported because "
                "minimum grouped binary holdout support is absent."
            )
    })

baseline_limitations = pd.DataFrame(
    limitations_records
)

display(
    baseline_limitations
)


,target,issue,value,implication
0,H6_violation,VERY_SMALL_LABELED_SET,12,Metrics have high variance and must be treated as preliminary.
1,H6_violation,PROXY_FEATURE_SPACE,EXPERIMENTAL_CONSTRUCT_PROXY,"Features are construct-level proxies, not strict canonical Nielsen metrics."
2,H6_violation,MIXED_SOURCE_MATCH_LEVEL,A_CONFIRMED_STRONG | B_MODERATE_VALIDATED | C_EXACT_NAME_TECHNICAL,Observed labels include source matches with different evidence strength.
3,H6_violation,UNLABELED_DATA_USED_IN_BASELINE,False,This is intentional; U is reserved for the semi-supervised experiment.
4,H1_violation,TARGET_BLOCKED,True,No supervised baseline is reported because minimum grouped binary holdout support is absent.
5,H4_violation,TARGET_BLOCKED,True,No supervised baseline is reported because minimum grouped binary holdout support is absent.
6,H5_violation,TARGET_BLOCKED,True,No supervised baseline is reported because minimum grouped binary holdout support is absent.


## 15. Criterio para continuar al notebook 29

El notebook 29 podrá probar aprendizaje semisupervisado únicamente para targets con baseline válido.

Para el estado actual:

```text
H6:
baseline supervisado → sí
SSL                  → candidato

H1/H4/H5:
baseline supervisado → bloqueado
SSL validado         → bloqueado
```

El experimento SSL deberá reutilizar exactamente los folds congelados aquí.


In [15]:
rf_rows = aggregate_metrics[
    aggregate_metrics[
        "model"
    ]
    ==
    "RANDOM_FOREST"
].copy()

continuation_records = []

for target in ALL_TARGETS:

    if target in ready_targets:

        metrics_row = rf_rows[
            rf_rows[
                "target"
            ]
            ==
            target
        ]

        continuation_records.append({
            "target":
                target,

            "supervised_baseline_available":
                not metrics_row.empty,

            "semi_supervised_experiment_allowed":
                not metrics_row.empty,

            "reason":
                (
                    "Grouped supervised baseline "
                    "completed."
                )
        })

    else:

        continuation_records.append({
            "target":
                target,

            "supervised_baseline_available":
                False,

            "semi_supervised_experiment_allowed":
                False,

            "reason":
                (
                    "Insufficient observed class/group "
                    "support in L."
                )
        })

continuation_decision = pd.DataFrame(
    continuation_records
)

display(
    continuation_decision
)


,target,supervised_baseline_available,semi_supervised_experiment_allowed,reason
0,H1_violation,False,False,Insufficient observed class/group support in L.
1,H4_violation,False,False,Insufficient observed class/group support in L.
2,H5_violation,False,False,Insufficient observed class/group support in L.
3,H6_violation,True,True,Grouped supervised baseline completed.


## 16. Guardar artefactos

In [16]:
fold_registry.to_csv(
    SEMISUP_REPORTS_DIR
    / "baseline_grouped_fold_summary.csv",
    index=False
)

# Registro fila-a-fold para reutilizar en SSL.
assignment_frames = []

for target, info in (
    fold_assignments.items()
):

    df = info[
        "data"
    ].copy()

    assignment_frames.append(
        df[
            [
                "master_instance_id",
                "group_id",
                target,
                "evaluation_fold"
            ]
        ].rename(
            columns={
                target:
                    "observed_y"
            }
        ).assign(
            target=target
        )
    )

baseline_fold_assignments = pd.concat(
    assignment_frames,
    ignore_index=True
)

baseline_fold_assignments.to_csv(
    SEMISUP_REPORTS_DIR
    / "baseline_grouped_fold_assignments.csv",
    index=False
)

oof_predictions.to_csv(
    SEMISUP_REPORTS_DIR
    / "baseline_oof_predictions.csv",
    index=False
)

fold_metrics.to_csv(
    SEMISUP_REPORTS_DIR
    / "baseline_fold_metrics.csv",
    index=False
)

aggregate_metrics.to_csv(
    SEMISUP_REPORTS_DIR
    / "baseline_aggregate_metrics.csv",
    index=False
)

baseline_comparison.to_csv(
    SEMISUP_REPORTS_DIR
    / "baseline_rf_vs_dummy.csv",
    index=False
)

feature_importance.to_csv(
    SEMISUP_REPORTS_DIR
    / "baseline_feature_importance.csv",
    index=False
)

final_model_registry.to_csv(
    SEMISUP_REPORTS_DIR
    / "baseline_final_model_registry.csv",
    index=False
)

provenance_by_class.to_csv(
    SEMISUP_REPORTS_DIR
    / "baseline_provenance_by_class.csv",
    index=False
)

baseline_limitations.to_csv(
    SEMISUP_REPORTS_DIR
    / "baseline_limitations.csv",
    index=False
)

continuation_decision.to_csv(
    SEMISUP_REPORTS_DIR
    / "baseline_continuation_decision.csv",
    index=False
)

with open(
    SEMISUP_REPORTS_DIR
    / "baseline_feature_contract.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        {
            "features":
                FEATURES,

            "threshold":
                PREDICTION_THRESHOLD,

            "random_state":
                RANDOM_STATE,

            "ready_targets":
                ready_targets,

            "blocked_targets":
                blocked_targets,

            "feature_space":
                "EXPERIMENTAL_CONSTRUCT_PROXY"
        },
        file,
        indent=2
    )


In [17]:
expected_outputs = [
    "baseline_grouped_fold_summary.csv",
    "baseline_grouped_fold_assignments.csv",
    "baseline_oof_predictions.csv",
    "baseline_fold_metrics.csv",
    "baseline_aggregate_metrics.csv",
    "baseline_rf_vs_dummy.csv",
    "baseline_feature_importance.csv",
    "baseline_final_model_registry.csv",
    "baseline_provenance_by_class.csv",
    "baseline_limitations.csv",
    "baseline_continuation_decision.csv",
    "baseline_feature_contract.json"
]

for filename in expected_outputs:

    path = (
        SEMISUP_REPORTS_DIR
        /
        filename
    )

    print(
        filename,
        "✅"
        if path.exists()
        else "❌"
    )

for target in ready_targets:

    model_path = (
        MODEL_DIR
        /
        f"{target}_supervised_rf.joblib"
    )

    print(
        model_path.relative_to(
            PROJECT_DIR
        ),
        "✅"
        if model_path.exists()
        else "❌"
    )


baseline_grouped_fold_summary.csv ✅
baseline_grouped_fold_assignments.csv ✅
baseline_oof_predictions.csv ✅
baseline_fold_metrics.csv ✅
baseline_aggregate_metrics.csv ✅
baseline_rf_vs_dummy.csv ✅
baseline_feature_importance.csv ✅
baseline_final_model_registry.csv ✅
baseline_provenance_by_class.csv ✅
baseline_limitations.csv ✅
baseline_continuation_decision.csv ✅
baseline_feature_contract.json ✅
models/semisupervised/baseline/H6_violation_supervised_rf.joblib ✅


## Punto de detención

Después de ejecutar este notebook, revisar antes de construir el notebook 29:

```text
baseline_aggregate_metrics.csv
baseline_rf_vs_dummy.csv
baseline_provenance_by_class.csv
baseline_continuation_decision.csv
```

El notebook 29 no debe asumir que el Random Forest supera al Dummy.

Si el baseline no muestra capacidad discriminativa, ese resultado debe conservarse y el experimento semisupervisado deberá responder si `U` aporta una mejora real.

El notebook 29 será:

```text
29_semisupervised_training.ipynb
```

y deberá:

1. reutilizar exactamente `baseline_grouped_fold_assignments.csv`;
2. mantener el fold de evaluación completamente fuera del pseudo-etiquetado;
3. usar únicamente el pool `U` del target;
4. registrar cada pseudo-etiqueta, probabilidad y ronda;
5. comparar contra el baseline supervisado sin cambiar el conjunto de evaluación.
